# Downloading the dataset
Download the Chesapeake Bay Land Cover dataset and organize your dataset directory as recommended.

The current notebook will check if there are any training images at data/cvpr/files/train. If so, the whole dataset will be considered as downloaded. If not, the three datasets "train", "test" and "val" will be downloaded. These downloads will be made with s5cmd. If not installed, this tool will be downloaded and installed (only for linux x86_64 machines for the moment)

Altertanively, you can download the datasets from a console running
```bash
# train
s5cmd --no-sign-request cp --include "*_lc.tif" --include "*_naip-new.tif" "s3://us-west-2.opendata.source.coop/agentmorris/lila-wildlife/lcmcvpr2019/cvpr_chesapeake_landcover/ny_1m_2013_extended-debuffered-train_tiles/*" data/cvpr/files/train/

# val
s5cmd --no-sign-request cp --include "*_lc.tif" --include "*_naip-new.tif" "s3://us-west-2.opendata.source.coop/agentmorris/lila-wildlife/lcmcvpr2019/cvpr_chesapeake_landcover/ny_1m_2013_extended-debuffered-val_tiles/*" data/cvpr/files/val/

# test
s5cmd --no-sign-request cp --include "*_lc.tif" --include "*_naip-new.tif" "s3://us-west-2.opendata.source.coop/agentmorris/lila-wildlife/lcmcvpr2019/cvpr_chesapeake_landcover/ny_1m_2013_extended-debuffered-test_tiles/*" data/cvpr/files/test/
```

Directory structure:
```
data/
└── cvpr/
    └── files/
        ├── train/
        ├── val/
        └── test/
```

In [ ]:
import subprocess
import platform
import os
from pathlib import Path

def is_s5cmd_installed():
    """Check if s5cmd is installed by trying to get its version."""
    try:
        result = subprocess.run(
            ["s5cmd", "version"],
            capture_output=True,
            text=True,
            check=True  # will raise CalledProcessError if exit code != 0
        )
        print("s5cmd found!")
    except FileNotFoundError:
        print("s5cmd is not installed or not in PATH.")
        if platform.system() == "Linux" and platform.machine() == "x86_64":
            print("Attempting to install s5cmd...")
            try:
                # Download s5cmd
                subprocess.run(["wget", "https://github.com/peak/s5cmd/releases/download/v2.3.0/s5cmd_2.3.0_Linux-64bit.tar.gz"], check=True)
                
                # Extract the archive
                subprocess.run(["tar", "-xvzf", "s5cmd_2.3.0_Linux-64bit.tar.gz"], check=True)
                
                # Create local bin directory if it doesn't exist
                os.makedirs(os.path.expanduser("~/bin"), exist_ok=True)
                
                # Move s5cmd to local bin
                subprocess.run(["mv", "s5cmd", os.path.expanduser("~/bin/")], check=True)
                
                # Update PATH
                os.environ["PATH"] = os.path.expanduser("~/bin") + ":" + os.environ["PATH"]
                
                # Verify installation
                result = subprocess.run(["s5cmd", "version"], capture_output=True, text=True, check=True)
                print("s5cmd successfully installed!")
                
            except subprocess.CalledProcessError as e:
                print(f"Error during installation: {e}")
                raise

def download_dataset(split):
    """
    Downloads the dataset for a specific split (train, val, or test)
    """
    # Create directory
    target_dir = Path(f"data/cvpr/files/{split}")
    target_dir.mkdir(parents=True, exist_ok=True)
    
    # Construct the s3 URL
    s3_url = f"s3://us-west-2.opendata.source.coop/agentmorris/lila-wildlife/lcmcvpr2019/cvpr_chesapeake_landcover/ny_1m_2013_extended-debuffered-{split}_tiles/*"
    
    try:
        # Run s5cmd command
        cmd = [
            "s5cmd", "--no-sign-request", "cp",
            "--include", "*_lc.tif",
            "--include", "*_naip-new.tif",
            s3_url,
            str(target_dir) + "/"
        ]
        
        print(f"\nDownloading {split} dataset...")
        result = subprocess.run(cmd, capture_output=True, text=True, check=True)
        print(f"Successfully downloaded {split} dataset")
        
    except subprocess.CalledProcessError as e:
        print(f"Error downloading {split} dataset:")
        print(f"Exit code: {e.returncode}")
        print(f"Error output: {e.stderr}")
        raise
    except Exception as e:
        print(f"Unexpected error downloading {split} dataset: {str(e)}")
        raise

In [ ]:
import os
from pathlib import Path
import sys

def find_project_root(marker='claymodel'):
    current = Path.cwd()
    for path in [current] + list(current.parents):
        if (path / marker).exists():
            return path
    raise FileNotFoundError(f"Project root not found")

os.chdir(find_project_root())
sys.path.append('.')

In [ ]:
TRAINING_DATA_DIR = "data/cvpr/files/train/"

training_data_path = Path(TRAINING_DATA_DIR)
if training_data_path.exists() and training_data_path.is_dir() and len(os.listdir(training_data_path)) > 0:
    print(f"✓ Files already exist at: {training_data_path.parent}")
else:
    print(f"✗ Files not found at: {training_data_path.parent}")
    print("Downloading dataset...")

    is_s5cmd_installed()

    for split in ['train', 'val', 'test']:
        try:
            download_dataset(split)
        except Exception as e:
            print(f"Failed to download {split} dataset. Error: {str(e)}")
            break  # Stop if any download fails
        print("-" * 50)


✗ Files not found at: data/cvpr/files
s5cmd found!
Version info: v2.3.0-991c9fb
Created directory: data/cvpr/files/train

Successfully downloaded train dataset
--------------------------------------------------
Created directory: data/cvpr/files/val

Successfully downloaded train dataset
--------------------------------------------------
Created directory: data/cvpr/files/val

Successfully downloaded val dataset
--------------------------------------------------
Created directory: data/cvpr/files/test

Successfully downloaded val dataset
--------------------------------------------------
Created directory: data/cvpr/files/test

Successfully downloaded test dataset
--------------------------------------------------
Successfully downloaded test dataset
--------------------------------------------------
